# Geometric Limits: Unit Circle, Pi Approximations, and Riemann Sums

This notebook investigates how the concept of a limit resolves geometric problems that cannot be answered with elementary Euclidean polygons alone—most notably, computing the **circumference** and **area** of a unit circle ($r = 1$).

## Outline

1. **Polygon Perimeter Limit:** Approximating circumference by regular $m$-gons ($L_m \to 2\pi$).
2. **Triangular Sector Area Limit:** Approximating area by summing triangular wedges ($A_m \to \pi$).
3. **Riemann Sums of Functions:** Integrating upper and lower semicircle functions $y = \pm \sqrt{1 - x^2}$.
4. **Concentric Annular Rings (Unwrapping):** Approximating circle area via thin cylindrical shells $\int_0^1 2\pi r \, dr = \pi$.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = [7.0, 5.0]
plt.rcParams["figure.autolayout"] = True

## 1. Circumference of a Unit Circle via Inscribed Polygons

Let a unit circle be partitioned into $m$ equal angular sectors. The angle subtended by each sector is:
$$\alpha = \frac{360^\circ}{m} = \frac{2\pi}{m} \text{ radians}$$

Using trigonometry, the chord segment (or half-segment height) is $h = r \sin(\alpha) = \sin(\alpha)$ when $r = 1$. The sum of all $m$ segments as $m \to \infty$ converges to the circumference $C = 2\pi$:

$$\lim_{m \to \infty} m \sin\left(\frac{2\pi}{m}\right) = 2\pi$$


In [ ]:
def compute_polygon_circumference(degree_step: float) -> float:
    """Compute perimeter of polygon with angular step in degrees."""
    sin_val = math.sin(math.radians(degree_step))
    num_parts = 360.0 / degree_step
    return sin_val * num_parts


print("Circumference approximation vs true 2*pi:")
for deg in [60.0, 30.0, 10.0, 1.0, 0.1, 0.01]:
    est = compute_polygon_circumference(deg)
    err = abs(est - 2.0 * math.pi)
    print(
        f"  step = {deg:6.2f} deg (parts = {360 / deg:7.0f}) -> Est: {est:10.7f} | Error: {err:10.2e}"
    )

assert abs(compute_polygon_circumference(0.01) - 2.0 * math.pi) < 1e-4

## 2. Unit Circle Area via Triangular Polygon Partition

Each triangular slice formed by radius $r = 1$ and angle $\alpha$ has base $b = \cos(\alpha)$ and height $h = \sin(\alpha)$. Its area is:
$$A_\alpha = \frac{1}{2} b h = \frac{1}{2} \sin(\alpha) \cos(\alpha)$$

Multiplying by $m = \frac{360^\circ}{\alpha}$ slices yields total area:
$$A_m = m \cdot \frac{1}{2} \sin(\alpha) \cos(\alpha)$$

As $\alpha \to 0$, $A_m \to \pi$.


In [ ]:
def compute_polygon_area(degree_step: float) -> float:
    rad = math.radians(degree_step)
    sin_deg = math.sin(rad)
    cos_deg = math.cos(rad)
    area_slice = 0.5 * sin_deg * cos_deg
    num_parts = 360.0 / degree_step
    return area_slice * num_parts


print("Area approximation via triangle partition vs math.pi:")
for deg in [45.0, 15.0, 5.0, 1.0, 0.1, 0.01]:
    est = compute_polygon_area(deg)
    err = abs(est - math.pi)
    print(f"  step = {deg:6.2f} deg -> Est: {est:10.7f} | Error: {err:10.2e}")

assert abs(compute_polygon_area(0.01) - math.pi) < 1e-4

## 3. Circle Area via Riemann Sum of Semicircular Functions

The equation of a unit circle is $x^2 + y^2 = 1$. Solving for $y$ yields two branches:
- Upper semicircle: $y = +\sqrt{1 - x^2}$
- Lower semicircle: $y = -\sqrt{1 - x^2}$

The area is twice the area under the upper semicircle:
$$\text{Area} = 2 \int_{-1}^{1} \sqrt{1 - x^2} \, dx \approx 2 \sum_{i=1}^n \sqrt{1 - x_i^2} \, \Delta x$$


In [ ]:
# Visualization of upper and lower semicircles
data_x = np.arange(-1.0, 1.0, 1e-4)
y_upper = np.sqrt(1.0 - data_x**2)
y_lower = -np.sqrt(1.0 - data_x**2)

plt.figure(figsize=(6, 6))
plt.plot(data_x, y_upper, "g-", label=r"Upper: $y = \sqrt{1 - x^2}$")
plt.plot(data_x, y_lower, "b-", label=r"Lower: $y = -\sqrt{1 - x^2}$")
plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.gca().set_aspect("equal")
plt.title("Unit Circle Boundary Decomposition")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def compute_riemann_circle_area(step: float) -> float:
    """Compute circle area using Riemann rectangle sum of 2 * sqrt(1 - x^2)."""
    x_grid = np.arange(-1.0, 1.0, step)
    # Avoid domain error at boundaries
    y_vals = np.sqrt(np.maximum(0.0, 1.0 - x_grid**2))
    half_area = np.sum(y_vals) * step
    return float(half_area * 2.0)


print("Riemann rectangle integration for circle area:")
for step in [0.1, 0.01, 1e-3, 1e-4, 1e-5]:
    est = compute_riemann_circle_area(step)
    err = abs(est - math.pi)
    print(f"  step = {step:8.5f} -> Estimated Area: {est:10.7f} | Error: {err:10.2e}")

assert abs(compute_riemann_circle_area(1e-4) - math.pi) < 1e-3

## 4. Concentric Annular Rings ("Unwrapping" Method)

Imagine peeling the circle into concentric rings of radius $r$ and thickness $dr$.
The circumference of each ring is $2\pi r$. When flattened, each ring forms a thin rectangle of area:
$$dA = 2\pi r \, dr$$

Summing these rings from $r = 0$ to $r = 1$:
$$\text{Area} = \int_0^1 2\pi r \, dr = \left[ \pi r^2 \right]_0^1 = \pi$$


In [ ]:
def compute_ring_area(dr: float) -> float:
    radii = np.arange(0.0, 1.0, dr)
    ring_areas = 2.0 * math.pi * radii * dr
    return float(np.sum(ring_areas))


print("Ring summation vs analytical pi:")
for dr in [0.1, 0.05, 0.01, 1e-3, 1e-4]:
    est = compute_ring_area(dr)
    err = abs(est - math.pi)
    print(f"  dr = {dr:8.5f} -> Ring Area: {est:10.7f} | Error: {err:10.2e}")

In [ ]:
# Visualization of the annular ring Riemann sum
dr = 0.05
r_vals = np.arange(0.0, 1.0, dr)
heights = 2.0 * math.pi * r_vals

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(
    r_vals,
    heights,
    width=dr,
    align="edge",
    color="skyblue",
    edgecolor="navy",
    alpha=0.7,
    label=r"Riemann Rings $2\pi r \, dr$",
)

# True continuous function f(r) = 2*pi*r
r_cont = np.linspace(0.0, 1.0, 200)
ax.plot(r_cont, 2.0 * math.pi * r_cont, "r-", lw=2, label=r"$f(r) = 2\pi r$")

ax.set_title(r"Unwrapping the Circle: $\int_0^1 2\pi r \, dr = \pi$")
ax.set_xlabel("Radius r")
ax.set_ylabel(r"Circumference $2\pi r$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Summary

- Approximating curved geometries through limit processes directly motivates integral calculus.
- Both spatial partitions (inscribed polygons, vertical Cartesian strips) and coordinate transforms (polar concentric shells) converge to the same invariant value $\pi$.
- In machine learning, continuous loss integrals over probability densities are routinely approximated using Monte Carlo and Riemann-like discrete sums over mini-batches.
